# AEGEAN | Kayak Scraping

Expand review cards, recover browser failures and merge checkpoints into the master dataset.

**Dependencies:** selenium beautifulsoup4 pandas. Install in the active kernel before running.

**Execution:** Run the setup, configuration and collection cells in order. Review airline selections and limits before collection. Existing data must be migrated to `Data/Scraping/kayak/` before resuming.

**Outputs:** All generated files are stored in `Data/Scraping/kayak/`. Existing filenames and schemas are retained.

Collection requires network access; browser-based scrapers also require Chrome.

## 1. Portable project paths

In [ ]:
from pathlib import Path

# Locate the project from the notebook kernel's working directory.
# Start Jupyter inside the project tree; the root name is not significant.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if all((p / name).is_dir() for name in ("Python Code", "Data", "Streamlit"))),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project folder containing Python Code, Data and Streamlit.")

SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
SOURCE_DIR = SCRAPING_DIR / 'kayak'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)


## Imports and prerequisites

In [ ]:
# ================================================================
# KAYAK MULTI-AIRLINE REVIEW SCRAPER
#
# Features:
# - Multiple airline URLs
# - Preserves existing master CSV
# - Skips completed airlines
# - Faster More-button expansion
# - Checkpoints every 50 new reviews
# - Automatic ChromeDriver recovery
# - Deduplication
# - Individual and combined CSV files
# - Progress and error logging
# ================================================================

import re
import os
import time
import random
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    WebDriverException,
    TimeoutException,
    StaleElementReferenceException
)

## 1. User Configuration

In [ ]:
# Add or remove URLs here.
# Use verified KAYAK URLs for each airline.

AIRLINE_URLS = {
    "Aegean Airlines": "https://www.kayak.com/Aegean-Airlines.A3.airline.html",
    "Turkish Airlines": "https://www.kayak.com/Turkish-Airlines.TK.airline.html",
    "Air France": "https://www.kayak.com/Air-France.AF.airline.html",
    "Lufthansa": "https://www.kayak.com/Lufthansa.LH.airline.html",
    "Bangkok Airways": "https://www.kayak.com/Bangkok-Airways.PG.airline.html",
    "Porter Airlines": "https://www.kayak.com/Porter-Airlines.PD.airline.html",
    "Virgin Australia": "https://www.kayak.com/Virgin-Australia.VA.airline.html",
    # Add other verified KAYAK airline URLs:
}

# Existing master CSV will be preserved.
MASTER_CSV = Path(SOURCE_DIR / "kayak_all_airline_reviews.csv")

STATUS_CSV = Path(SOURCE_DIR / "kayak_scraping_status.csv")

CHECKPOINT_DIR = Path(SOURCE_DIR / "kayak_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

DEBUG_DIR = Path(SOURCE_DIR / "kayak_debug")
DEBUG_DIR.mkdir(exist_ok=True)

# Explicitly skip Aegean, already collected.
SKIP_AIRLINES = {"Aegean Airlines"}

# Also skip airlines marked completed in the status CSV.
SKIP_COMPLETED = True

# Browser settings
HEADLESS = False
PAGE_LOAD_TIMEOUT = 75

# Delays
CLICK_DELAY_MIN = 0.5
CLICK_DELAY_MAX = 1.2

PAGE_DELAY_MIN = 3
PAGE_DELAY_MAX = 6

AIRLINE_DELAY_MIN = 4
AIRLINE_DELAY_MAX = 8

# Expansion settings
NEW_REVIEWS_PER_CHECKPOINT = 50
WAIT_FOR_REVIEWS = 5
MAX_NO_GROWTH = 4

# Safety limit; increase if necessary.
MAX_MORE_CLICKS = 3000

# Browser recovery
MAX_BROWSER_RETRIES = 3
RETRY_DELAY = 10

# Save expanded HTML at checkpoints.
# Set False to reduce disk usage.
SAVE_HTML_CHECKPOINTS = True

## 2. General Helpers

In [ ]:
def pause(low, high):
    """Sleep for a randomized interval within the supplied bounds."""
    time.sleep(random.uniform(low, high))


def clean(text):
    """Normalize whitespace in text."""
    return re.sub(r"\s+", " ", text or "").strip()


def safe_filename(value):
    """Convert a label into a filesystem-friendly name."""
    return re.sub(
        r"[^a-zA-Z0-9_-]+",
        "_",
        value
    ).strip("_").lower()


def utc_now():
    """Return the current UTC timestamp."""
    return datetime.now(
        timezone.utc
    ).isoformat(timespec="seconds")


def atomic_csv_save(df, path):
    """
    Write to a temporary file and then replace the target.
    This reduces the risk of corrupting the master CSV.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_name(
        path.name + ".tmp"
    )

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(temp_path, path)


def get_text(card, selector):
    """Extract text from the first matching HTML element."""
    node = card.select_one(selector)

    if node:
        return clean(node.get_text(" ", strip=True))

    return None

## 3. Create Chrome Driver

In [ ]:
def create_driver():

    """Configure and initialize the Chrome WebDriver."""
    options = Options()

    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=en-US")
    options.add_argument("--window-size=1500,1100")

    if HEADLESS:
        options.add_argument("--headless=new")

    # Selenium Manager automatically locates ChromeDriver.
    driver = webdriver.Chrome(options=options)

    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)

    return driver

## 4. Cookie Consent

In [ ]:
def accept_cookies(driver):

    """Attempt to dismiss the cookie consent interface."""
    labels = [
        "Accept all",
        "Accept",
        "Agree",
        "I agree",
        "Allow all"
    ]

    for label in labels:

        try:
            buttons = driver.find_elements(
                By.XPATH,
                f"//button[normalize-space(.)='{label}']"
            )

            for button in buttons:
                if button.is_displayed() and button.is_enabled():
                    button.click()
                    pause(0.5, 1)
                    return True

        except WebDriverException:
            continue

    return False

## 5. Locate Reviews Section

In [ ]:
def scroll_to_reviews(driver):

    """Navigate the browser to the review section."""
    xpath = (
        "//*[self::h2 or self::h3]"
        "[contains(translate(normalize-space(.),"
        "'ABCDEFGHIJKLMNOPQRSTUVWXYZ',"
        "'abcdefghijklmnopqrstuvwxyz'),"
        "'customer reviews')]"
    )

    headings = driver.find_elements(
        By.XPATH,
        xpath
    )

    if not headings:
        return False

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        headings[0]
    )

    pause(0.8, 1.5)

    return True

## 6. Fast Review Count

In [ ]:
def fast_review_count(driver):

    """Count review cards currently loaded in the browser."""
    return driver.execute_script(
        """
        return document.querySelectorAll('div.c2oma').length;
        """
    )

## 7. Fast More Button Detection

In [ ]:
def find_more_button_fast(driver):

    """Locate the control for loading additional reviews."""
    return driver.execute_script("""
        const heading = [...document.querySelectorAll('h2,h3')]
            .find(el =>
                el.textContent.toLowerCase()
                .includes('customer reviews')
            );

        if (!heading) return null;

        // Find a container that includes the review cards.
        let scope = heading.parentElement;

        while (
            scope &&
            !scope.querySelector('div.c2oma')
        ) {
            scope = scope.parentElement;
        }

        if (!scope) return null;

        const pattern =
            /^(more|show more|load more|see more|more reviews|show more reviews|load more reviews|see more reviews)$/i;

        const controls = [
            ...scope.querySelectorAll(
                'button,a,[role="button"]'
            )
        ];

        const matches = controls.filter(el => {
            const label = (
                el.innerText ||
                el.getAttribute('aria-label') ||
                el.getAttribute('title') ||
                ''
            ).trim();

            return (
                pattern.test(label) &&
                el.getClientRects().length > 0 &&
                !el.disabled &&
                el.getAttribute('aria-disabled') !== 'true'
            );
        });

        return matches.length
            ? matches[matches.length - 1]
            : null;
    """)

## 8. Review Parsing

In [ ]:
def parse_rating(text):

    """Parse the displayed review rating."""
    if not text:
        return None, None

    match = re.match(
        r"(\d+(?:\.\d+)?)\s*(.*)",
        text
    )

    if not match:
        return None, text

    return (
        float(match.group(1)),
        clean(match.group(2))
    )


def parse_user_info(text):

    """Extract reviewer metadata from the displayed text."""
    if not text:
        return None, None, None

    match = re.search(
        r",\s*([A-Za-z]+)\s+(\d{4})$",
        text
    )

    if not match:
        return clean(text), None, None

    author = clean(text[:match.start()])

    raw_date = (
        f"{match.group(1)} {match.group(2)}"
    )

    publication_date = None

    for fmt in ("%b %Y", "%B %Y"):

        try:
            publication_date = datetime.strptime(
                raw_date,
                fmt
            ).strftime("%Y-%m")

            break

        except ValueError:
            continue

    return author, publication_date, raw_date


def parse_route(card):

    """Extract available route information from a review card."""
    buttons = card.select(
        ".c2oma-route button"
    )

    if len(buttons) >= 2:

        origin = clean(
            buttons[0].get_text(" ", strip=True)
        )

        destination = clean(
            buttons[1].get_text(" ", strip=True)
        )

    else:
        origin = None
        destination = None

    route = (
        f"{origin}-{destination}"
        if origin and destination
        else None
    )

    return origin, destination, route

## 9. Extract Reviews

In [ ]:
REVIEW_COLUMNS = [
    "review_id",
    "airline",
    "source",
    "author",
    "publication_date",
    "publication_date_raw",
    "overall_rating",
    "rating_label",
    "origin",
    "destination",
    "route",
    "review_text",
    "raw_review_text",
    "source_url",
    "scraped_at"
]


def extract_reviews(html, airline, url):

    """Parse review cards into structured records."""
    soup = BeautifulSoup(html, "lxml")

    cards = soup.select("div.c2oma")

    reviews = []

    for card in cards:

        rating_raw = get_text(
            card, ".c2oma-rating"
        )

        rating, rating_label = parse_rating(
            rating_raw
        )

        user_raw = get_text(
            card, ".c2oma-user-info"
        )

        author, publication_date, publication_date_raw = (
            parse_user_info(user_raw)
        )

        origin, destination, route = parse_route(card)

        review_text = get_text(
            card, ".c2oma-review-text"
        )

        raw_review_text = clean(
            card.get_text(" ", strip=True)
        )

        # Preserve compatibility with previous review IDs.
        identity = "|".join([
            airline,
            str(author),
            str(publication_date),
            str(route),
            str(rating),
            str(review_text)
        ])

        review_id = hashlib.sha256(
            identity.encode("utf-8")
        ).hexdigest()[:20]

        reviews.append({
            "review_id": review_id,
            "airline": airline,
            "source": "KAYAK",
            "author": author,
            "publication_date": publication_date,
            "publication_date_raw": publication_date_raw,
            "overall_rating": rating,
            "rating_label": rating_label,
            "origin": origin,
            "destination": destination,
            "route": route,
            "review_text": review_text,
            "raw_review_text": raw_review_text,
            "source_url": url,
            "scraped_at": utc_now()
        })

    df = pd.DataFrame(
        reviews,
        columns=REVIEW_COLUMNS
    )

    if not df.empty:
        df = df.drop_duplicates(
            subset=["review_id"]
        ).reset_index(drop=True)

    return df

## 10. Master Csv Management

In [ ]:
def load_master():

    """Read the existing combined review dataset."""
    if MASTER_CSV.exists():

        df = pd.read_csv(
            MASTER_CSV,
            dtype={"review_id": str},
            encoding="utf-8-sig"
        )

        print(
            f"Existing master CSV: {len(df)} reviews"
        )

        return df

    return pd.DataFrame(columns=REVIEW_COLUMNS)


def merge_reviews(new_df):

    """
    Read the latest master CSV from disk each time.
    This prevents stale in-memory data from overwriting
    previously saved reviews.
    """

    master = load_master()

    if new_df.empty:
        return master

    combined = pd.concat(
        [master, new_df],
        ignore_index=True
    )

    combined = combined.drop_duplicates(
        subset=["review_id"],
        keep="first"
    )

    combined = combined.reset_index(drop=True)

    atomic_csv_save(
        combined,
        MASTER_CSV
    )

    return combined

## 11. Checkpoint Saving

In [ ]:
def save_checkpoint(driver, airline, url):

    """Persist current browser content and extracted review records."""
    name = safe_filename(airline)

    html = driver.page_source

    df = extract_reviews(
        html,
        airline,
        url
    )

    if SAVE_HTML_CHECKPOINTS:

        html_path = (
            CHECKPOINT_DIR / f"{name}_latest.html"
        )

        temp_html = html_path.with_name(
            html_path.name + ".tmp"
        )

        temp_html.write_text(
            html,
            encoding="utf-8"
        )

        os.replace(temp_html, html_path)

    if df.empty:
        print("WARNING: Checkpoint contains no reviews.")
        return df

    # Save airline checkpoint
    checkpoint_csv = (
        CHECKPOINT_DIR / f"{name}_latest.csv"
    )

    atomic_csv_save(
        df,
        checkpoint_csv
    )

    # Save individual airline file
    individual_csv = Path(
        SOURCE_DIR / f"kayak_{name}_reviews.csv"
    )

    # Preserve existing airline records.
    if individual_csv.exists():

        old = pd.read_csv(
            individual_csv,
            dtype={"review_id": str}
        )

        df = pd.concat(
            [old, df],
            ignore_index=True
        ).drop_duplicates(
            subset=["review_id"],
            keep="first"
        )

    atomic_csv_save(
        df,
        individual_csv
    )

    # Merge into master immediately
    master = merge_reviews(df)

    airline_count = (
        master["airline"] == airline
    ).sum()

    print(
        f"CHECKPOINT SAVED | {airline} | "
        f"{airline_count} total reviews in master"
    )

    return df

## 12. Status Tracking

In [ ]:
def load_status():

    """Read the persisted scraping status table."""
    if STATUS_CSV.exists():
        return pd.read_csv(STATUS_CSV)

    return pd.DataFrame()


def record_status(
    airline,
    url,
    status,
    reviews,
    clicks,
    reason,
    error=None
):

    """Update the status table for an airline run."""
    row = pd.DataFrame([{
        "airline": airline,
        "url": url,
        "status": status,
        "reviews_extracted": reviews,
        "more_clicks": clicks,
        "stop_reason": reason,
        "error": str(error)[:500] if error else None,
        "timestamp": utc_now()
    }])

    old = load_status()

    combined = pd.concat(
        [old, row],
        ignore_index=True
    )

    atomic_csv_save(
        combined,
        STATUS_CSV
    )


def get_completed_airlines():

    """Identify airlines already marked as completed."""
    status = load_status()

    if status.empty:
        return set()

    latest = status.drop_duplicates(
        subset=["airline"],
        keep="last"
    )

    return set(
        latest.loc[
            latest["status"] == "completed",
            "airline"
        ]
    )

## 13. Fast Expansion With Checkpoints

In [ ]:
def expand_reviews(driver, airline, url):

    """Load additional review cards and save periodic checkpoints."""
    clicks = 0
    no_growth = 0

    previous_count = fast_review_count(driver)

    last_saved_count = 0

    print(
        f"Initial review cards: {previous_count}"
    )

    stop_reason = None

    while clicks < MAX_MORE_CLICKS:

        button = find_more_button_fast(driver)

        if button is None:
            stop_reason = "No visible More button"
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            button
        )

        button.click()

        clicks += 1

        try:

            WebDriverWait(
                driver,
                WAIT_FOR_REVIEWS,
                poll_frequency=0.2
            ).until(
                lambda d:
                fast_review_count(d) > previous_count
            )

        except TimeoutException:
            pass

        current_count = fast_review_count(driver)

        if current_count > previous_count:

            previous_count = current_count
            no_growth = 0

        else:

            no_growth += 1

            print(
                f"No growth at click {clicks}: "
                f"{no_growth}/{MAX_NO_GROWTH}"
            )

            if no_growth >= MAX_NO_GROWTH:
                stop_reason = "Repeated no growth"
                break

        # Save after every 50 newly loaded reviews.
        if (
            current_count - last_saved_count
            >= NEW_REVIEWS_PER_CHECKPOINT
        ):

            save_checkpoint(
                driver,
                airline,
                url
            )

            last_saved_count = current_count

        if clicks % 10 == 0:

            print(
                f"Click {clicks} | "
                f"Loaded reviews: {current_count}"
            )

        pause(
            CLICK_DELAY_MIN,
            CLICK_DELAY_MAX
        )

    if clicks >= MAX_MORE_CLICKS:
        stop_reason = "Maximum click limit reached"

    # Always save the final state.
    final_df = save_checkpoint(
        driver,
        airline,
        url
    )

    return final_df, {
        "clicks": clicks,
        "reviews": len(final_df),
        "stop_reason": stop_reason
    }

## 14. Scrape One Airline

In [ ]:
def scrape_airline(driver, airline, url):

    """Collect reviews for one airline using the configured scraper."""
    print("\n" + "=" * 75)
    print("AIRLINE:", airline)
    print("URL:", url)
    print("=" * 75)

    driver.get(url)

    pause(
        PAGE_DELAY_MIN,
        PAGE_DELAY_MAX
    )

    accept_cookies(driver)

    found = scroll_to_reviews(driver)

    if not found:
        raise RuntimeError(
            "Customer reviews section not found"
        )

    WebDriverWait(
        driver,
        30
    ).until(
        lambda d: fast_review_count(d) > 0
    )

    return expand_reviews(
        driver,
        airline,
        url
    )

## 15. Automatic Browser Recovery

In [ ]:
def scrape_with_recovery(airline, url):

    """Retry airline collection after recoverable browser failures."""
    last_error = None

    for attempt in range(
        1,
        MAX_BROWSER_RETRIES + 1
    ):

        driver = None

        print(
            f"\nAttempt {attempt}/"
            f"{MAX_BROWSER_RETRIES}: {airline}"
        )

        try:

            driver = create_driver()

            df, info = scrape_airline(
                driver,
                airline,
                url
            )

            is_complete = (
                info["stop_reason"]
                == "No visible More button"
            )

            status = (
                "completed"
                if is_complete
                else "partial"
            )

            master = load_master()

            total_saved = (
                (master["airline"] == airline).sum()
                if "airline" in master.columns
                else len(df)
            )

            record_status(
                airline=airline,
                url=url,
                status=status,
                reviews=total_saved,
                clicks=info["clicks"],
                reason=info["stop_reason"]
            )

            print(
                f"\n{airline}: {status} | "
                f"{total_saved} reviews saved"
            )

            # Retry partial expansions only when they
            # ended due to an apparent transient problem.
            if status == "partial":

                last_error = RuntimeError(
                    info["stop_reason"]
                )

                if (
                    info["stop_reason"]
                    == "Maximum click limit reached"
                ):
                    return False

            else:
                return True

        except Exception as error:

            last_error = error

            print(
                f"Browser/scraping error: "
                f"{str(error)[:250]}"
            )

            # Try to save any remaining live DOM.
            if driver is not None:

                try:
                    save_checkpoint(
                        driver,
                        airline,
                        url
                    )
                except Exception:
                    print(
                        "Browser unavailable. "
                        "Using previous disk checkpoints."
                    )

            master = load_master()

            total_saved = (
                (master["airline"] == airline).sum()
                if "airline" in master.columns
                else 0
            )

            record_status(
                airline=airline,
                url=url,
                status="partial",
                reviews=total_saved,
                clicks=None,
                reason="Browser error",
                error=error
            )

        finally:

            if driver is not None:
                try:
                    driver.quit()
                except Exception:
                    pass

        if attempt < MAX_BROWSER_RETRIES:

            print(
                f"Restarting browser in "
                f"{RETRY_DELAY} seconds..."
            )

            time.sleep(RETRY_DELAY)

    print(
        f"\nFAILED/PARTIAL: {airline}"
    )

    print("Last error:", last_error)

    return False

## 16. Main

In [ ]:
def main():

    """Orchestrate collection and report the resulting dataset."""
    master = load_master()

    print("\nExisting reviews by airline:")

    if not master.empty:
        print(
            master["airline"].value_counts()
        )

    completed = (
        get_completed_airlines()
        if SKIP_COMPLETED
        else set()
    )

    results = []

    for index, (airline, url) in enumerate(
        AIRLINE_URLS.items(),
        start=1
    ):

        print("\n" + "=" * 75)
        print(
            f"Processing airline "
            f"{index}/{len(AIRLINE_URLS)}"
        )
        print("=" * 75)

        if airline in SKIP_AIRLINES:

            print(
                f"SKIPPING {airline}: "
                "explicitly excluded"
            )

            continue

        if airline in completed:

            print(
                f"SKIPPING {airline}: "
                "already completed"
            )

            continue

        success = scrape_with_recovery(
            airline,
            url
        )

        results.append({
            "airline": airline,
            "completed": success
        })

        pause(
            AIRLINE_DELAY_MIN,
            AIRLINE_DELAY_MAX
        )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    master = load_master()

    print("\n" + "=" * 75)
    print("FINAL SCRAPING SUMMARY")
    print("=" * 75)

    print(
        "Total unique reviews:",
        len(master)
    )

    if not master.empty:

        print("\nReviews by airline:")

        print(
            master.groupby("airline")
            .agg(
                reviews=("review_id", "count"),
                average_rating=("overall_rating", "mean")
            )
            .round(2)
            .to_string()
        )

    print("\nMaster CSV:", MASTER_CSV)
    print("Status CSV:", STATUS_CSV)
    print("Checkpoint folder:", CHECKPOINT_DIR)

    return master

## 17. Run

In [ ]:
df_reviews = main()

display(df_reviews.head(20))

## Inspect saved data / checkpoint status

In [ ]:
from pathlib import Path

files = [
    SOURCE_DIR / "kayak_all_airline_reviews.csv",
    SOURCE_DIR / "kayak_turkish_airlines_reviews.csv",
    SOURCE_DIR / "kayak_debug/turkish_airlines_expanded.html",
    SOURCE_DIR / "kayak_debug/turkish_airlines_checkpoint.html",
    SOURCE_DIR / "kayak_checkpoints/turkish_airlines_latest.csv",
    SOURCE_DIR / "kayak_checkpoints/turkish_airlines_latest.html",
]

for filename in files:
    path = Path(filename)

    if path.exists():
        print(f"FOUND: {filename}")
        print(f"Size: {path.stat().st_size:,} bytes")
    else:
        print(f"NOT FOUND: {filename}")

## Inspect saved data / checkpoint status

In [ ]:
import pandas as pd

df = pd.read_csv(SOURCE_DIR / "kayak_all_airline_reviews.csv")

print("Total reviews:", len(df))
print("\nReviews by airline:")
print(df["airline"].value_counts())

## Inspect saved data / checkpoint status

In [ ]:
df.head()